# 02 - Run Evaluations

This notebook runs LLM evaluations on the Backtest Lie Detector benchmark.

## Prerequisites
1. Configure API keys in `.env` file
2. Generate benchmark with `01_build_benchmark.ipynb`

## Contents
1. Configure evaluation settings
2. Run evaluations
3. Monitor progress
4. Export results

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
from pathlib import Path
from dotenv import load_dotenv

from backtest_lie_detector.schemas import EvaluationConfig
from backtest_lie_detector.evals.run_eval import (
    run_evaluation,
    load_benchmark,
    load_results,
)
from backtest_lie_detector.evals.scoring import (
    aggregate_scores,
    scores_to_dataframe,
)

# Load environment variables
load_dotenv('../.env')

# Check for API keys
has_openai = bool(os.getenv('OPENAI_API_KEY'))
has_anthropic = bool(os.getenv('ANTHROPIC_API_KEY'))

print(f"OpenAI API key: {'✓' if has_openai else '✗'}")
print(f"Anthropic API key: {'✓' if has_anthropic else '✗'}")

## 1. Configure Evaluation Settings

In [ ]:
# Define evaluation configurations
configs = []

# GPT-4o with finance auditor prompt
if has_openai:
    configs.append(EvaluationConfig(
        model_name="gpt-4o",
        config_name="gpt-4o_finance_auditor",
        provider="openai",
        system_prompt_type="finance_auditor",
        temperature=0.0,
        max_tokens=2000,
        few_shot_examples=0,
    ))
    
    # GPT-4o with minimal prompt (baseline)
    configs.append(EvaluationConfig(
        model_name="gpt-4o",
        config_name="gpt-4o_minimal",
        provider="openai",
        system_prompt_type="minimal",
        temperature=0.0,
        max_tokens=2000,
        few_shot_examples=0,
    ))

# Claude Sonnet with finance auditor prompt
if has_anthropic:
    configs.append(EvaluationConfig(
        model_name="claude-sonnet-4-20250514",
        config_name="claude-sonnet_finance_auditor",
        provider="anthropic",
        system_prompt_type="finance_auditor",
        temperature=0.0,
        max_tokens=2000,
        few_shot_examples=0,
    ))

print(f"Configured {len(configs)} evaluation settings:")
for c in configs:
    print(f"  - {c.config_name}")

In [ ]:
# If no API keys, use mock client for demonstration
if not configs:
    print("No API keys found. Adding mock configuration for demonstration.")
    configs.append(EvaluationConfig(
        model_name="mock-model",
        config_name="mock_test",
        provider="openai",  # Ignored for mock
        system_prompt_type="finance_auditor",
        temperature=0.0,
        max_tokens=2000,
    ))

## 2. Run Evaluations

In [ ]:
# Paths
BENCHMARK_PATH = "../data/benchmark/benchmark_v1.jsonl"
# Use sample for quick testing
# BENCHMARK_PATH = "../data/benchmark/benchmark_sample.jsonl"
OUTPUT_PATH = "../outputs/results/model_outputs.jsonl"

# Load benchmark to check size
cases = load_benchmark(BENCHMARK_PATH)
print(f"Benchmark size: {len(cases)} cases")
print(f"Expected API calls: {len(cases) * len(configs)}")

In [ ]:
# Run evaluation (this may take a while)
# Set rate_limit_delay based on your API tier

results = run_evaluation(
    benchmark_path=BENCHMARK_PATH,
    output_path=OUTPUT_PATH,
    configs=configs,
    resume=True,  # Resume from existing results
    rate_limit_delay=1.0,  # 1 second between calls
    verbose=True,
)

## 3. Check Results

In [ ]:
# Load all results
all_scores = load_results(OUTPUT_PATH)
print(f"Total scored responses: {len(all_scores)}")

# Convert to DataFrame
scores_df = scores_to_dataframe(all_scores)
scores_df.head()

In [ ]:
# Quick summary by config
summary = scores_df.groupby('config_name').agg({
    'parse_success': 'mean',
    'validity_correct': 'mean',
    'violation_recall': 'mean',
    'violation_precision': 'mean',
    'repair_score': 'mean',
}).round(3)

summary.columns = ['Parse Rate', 'Validity Acc', 'Violation Recall', 'Violation Prec', 'Repair Score']
summary

## 4. Save Summary

Results are automatically saved to `outputs/results/model_outputs.jsonl`.

Continue to `03_analyze_results.ipynb` for detailed analysis and visualizations.

In [ ]:
# Save summary CSV
scores_df.to_csv('../outputs/results/scores.csv', index=False)
print("Saved scores to outputs/results/scores.csv")